# Transfer Learning for Text: From Language Model to Sentiment Classifier

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/transfer_learning_nlp.ipynb)

Build a sentiment classifier by fine-tuning a pretrained language model on IMDb reviews. Implements tokenisation, gradual unfreezing, and discriminative learning rates.

**Blog post:** [sesen.ai/blog/transfer-learning-nlp-language-model-sentiment](https://sesen.ai/blog/transfer-learning-nlp-language-model-sentiment)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
import numpy as np
import matplotlib.pyplot as plt

## Stage 1: Load a Pretrained Language Model

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
pretrained = AutoModel.from_pretrained(model_name)

print(f"Model: {model_name}")
print(f"Parameters: {sum(p.numel() for p in pretrained.parameters()):,}")
print(f"Vocabulary: {tokenizer.vocab_size:,} tokens")

# See what the model "knows" about language
text = "The movie was absolutely [MASK] and I loved every minute of it."
inputs = tokenizer(text, return_tensors="pt")
outputs = pretrained(**inputs)
print(f"Hidden state shape: {outputs.last_hidden_state.shape}")

## Stage 2: Prepare IMDb Data

In [ ]:
from datasets import load_dataset

imdb = load_dataset("imdb")
print(f"Train: {len(imdb['train'])} reviews")
print(f"Test:  {len(imdb['test'])} reviews")
print(f"Example: {imdb['train'][0]['text'][:200]}...")
print(f"Label: {'positive' if imdb['train'][0]['label'] == 1 else 'negative'}")

In [ ]:
class IMDbDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = IMDbDataset(imdb['train']['text'], imdb['train']['label'], tokenizer)
test_dataset = IMDbDataset(imdb['test']['text'], imdb['test']['label'], tokenizer)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

## Tokenisation: From Text to Numbers

In [ ]:
text = "The cinematography was breathtakingly beautiful"
tokens = tokenizer.tokenize(text)
ids = tokenizer.encode(text)
print(f"Tokens: {tokens}")
print(f"IDs:    {ids}")

## Stage 3: Build Classifier with Gradual Unfreezing

In [ ]:
class SentimentClassifier(nn.Module):
    """Pretrained language model + classification head."""

    def __init__(self, pretrained_model, num_classes=2, hidden_dim=768):
        super().__init__()
        self.encoder = pretrained_model
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Use [CLS] token representation (first token)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_output)

model = SentimentClassifier(pretrained)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Classifier head:  {sum(p.numel() for p in model.classifier.parameters()):,}")

## Gradual Unfreezing

In [ ]:
def freeze_encoder(model):
    """Freeze all encoder parameters."""
    for param in model.encoder.parameters():
        param.requires_grad = False

def unfreeze_last_n_layers(model, n):
    """Unfreeze the last n transformer layers."""
    layers = list(model.encoder.transformer.layer)
    # First freeze everything
    for param in model.encoder.parameters():
        param.requires_grad = False
    # Then unfreeze last n layers
    for layer in layers[-n:]:
        for param in layer.parameters():
            param.requires_grad = True

def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Stage 1: Train only the classifier head
freeze_encoder(model)
print(f"Head only: {count_trainable(model):,} trainable params")

# Stage 2: Unfreeze last 2 transformer layers
unfreeze_last_n_layers(model, 2)
print(f"Last 2 layers: {count_trainable(model):,} trainable params")

# Stage 3: Unfreeze everything
for param in model.parameters():
    param.requires_grad = True
print(f"All unfrozen: {count_trainable(model):,} trainable params")

## Discriminative Learning Rates

In [ ]:
def get_discriminative_lrs(base_lr, model, decay_factor=2.6):
    """Assign lower LRs to deeper layers, higher to the head."""
    param_groups = []

    # Encoder layers: progressively higher LR
    layers = list(model.encoder.transformer.layer)
    n_layers = len(layers)
    for i, layer in enumerate(layers):
        lr = base_lr / (decay_factor ** (n_layers - i))
        param_groups.append({'params': list(layer.parameters()), 'lr': lr})

    # Classifier head: full learning rate
    param_groups.append({'params': list(model.classifier.parameters()), 'lr': base_lr})

    return param_groups

# Example: base_lr=1e-3
param_groups = get_discriminative_lrs(1e-3, model)
for i, pg in enumerate(param_groups):
    print(f"Group {i}: lr={pg['lr']:.6f}")

In [ ]:
# Visualise discriminative LRs
lrs = [pg['lr'] for pg in param_groups]
labels = [f'Layer {i}' for i in range(len(lrs)-1)] + ['Head']

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#3b82f6'] * (len(lrs)-1) + ['#ef4444']
ax.bar(labels, lrs, color=colors)
ax.set_ylabel('Learning Rate', fontsize=12)
ax.set_title('Discriminative Learning Rates (base_lr=1e-3, decay=2.6)', fontsize=14)
ax.set_yscale('log')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## The Full Training Loop

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
model = model.to(device)

def train_epoch(model, loader, optimiser, device='cpu'):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        logits = model(input_ids, attention_mask)
        loss = F.cross_entropy(logits, labels)

        optimiser.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()

        total_loss += loss.item() * len(labels)
        correct += (logits.argmax(1) == labels).sum().item()
        total += len(labels)

    return total_loss / total, correct / total * 100

def evaluate(model, loader, device='cpu'):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            logits = model(input_ids, attention_mask)
            correct += (logits.argmax(1) == labels).sum().item()
            total += len(labels)
    return correct / total * 100

In [ ]:
stage_results = []

# Stage 1: Train head only (1 epoch)
freeze_encoder(model)
optimiser = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=2e-3
)
loss, acc = train_epoch(model, train_loader, optimiser, device)
test_acc = evaluate(model, test_loader, device)
print(f"Head only:      train={acc:.1f}%, test={test_acc:.1f}%")
stage_results.append(('Head only', acc, test_acc))

# Stage 2: Unfreeze last 2 layers (1 epoch, lower LR)
unfreeze_last_n_layers(model, 2)
optimiser = torch.optim.AdamW(
    get_discriminative_lrs(1e-3, model), weight_decay=0.01
)
loss, acc = train_epoch(model, train_loader, optimiser, device)
test_acc = evaluate(model, test_loader, device)
print(f"Last 2 layers:  train={acc:.1f}%, test={test_acc:.1f}%")
stage_results.append(('Last 2 layers', acc, test_acc))

# Stage 3: Unfreeze all (2 epochs, even lower LR)
for param in model.parameters():
    param.requires_grad = True
optimiser = torch.optim.AdamW(
    get_discriminative_lrs(5e-4, model), weight_decay=0.01
)
for epoch in range(2):
    loss, acc = train_epoch(model, train_loader, optimiser, device)
    test_acc = evaluate(model, test_loader, device)
    print(f"Full unfreeze {epoch+1}: train={acc:.1f}%, test={test_acc:.1f}%")
    stage_results.append((f'Full unfreeze {epoch+1}', acc, test_acc))

In [ ]:
# Plot training stages
stages = [r[0] for r in stage_results]
train_accs = [r[1] for r in stage_results]
test_accs = [r[2] for r in stage_results]

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(stages))
ax.plot(x, train_accs, 'o-', color='#3b82f6', linewidth=2, markersize=10, label='Train')
ax.plot(x, test_accs, 's-', color='#ef4444', linewidth=2, markersize=10, label='Test')
ax.set_xticks(list(x))
ax.set_xticklabels(stages, fontsize=10)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Gradual Unfreezing: Accuracy at Each Stage', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(80, 100)
plt.tight_layout()
plt.show()

## Exercises

1. **No gradual unfreezing** — Unfreeze everything from the start and train for 4 epochs with the same total compute. Compare final accuracy to the gradual approach.

2. **More stages** — Try unfreezing 1 layer at a time (6 stages instead of 3). Does the extra granularity help?

3. **Smaller dataset** — Subsample 1000 training examples. How much does transfer learning help compared to training from scratch?

4. **Different base model** — Replace DistilBERT with `bert-base-uncased` (larger). Does the extra capacity improve accuracy?

5. **Decay factor** — Try decay factors of 1.5, 2.6, and 5.0 for discriminative LRs. Which gives the best test accuracy?

## References

- Howard, J. & Ruder, S. (2018). [Universal Language Model Fine-tuning for Text Classification.](https://arxiv.org/abs/1801.06146)
- Devlin, J. et al. (2018). [BERT: Pre-training of Deep Bidirectional Transformers.](https://arxiv.org/abs/1810.04805)
- fast.ai course: [Practical Deep Learning for Coders, Lesson 3](https://course.fast.ai/).